# Projeto 4 — Clipping comparativo (peso 2)

**Pergunta:** o que este recorte de notícias da Agência Brasil e da CNN Brasil mostra sobre a cobertura de Lula e Flávio Bolsonaro? Transforme textos em dados auditáveis e escreva uma conclusão proporcional ao tamanho da amostra. A base tem 40 matérias selecionadas, dez por combinação de veículo e foco, publicadas em outubro de 2026. **Não é uma amostra aleatória de toda a cobertura.**

Entregue uma pasta `23-projeto-4-noticias/` no seu repositório privado com notebook executado, `README.md` e tabelas de resultado pequenas. O projeto integra Aulas 21–23 e vale peso 2. Crie sua **própria chave Groq**; nunca a salve nem a envie.

## 0. Carregar e conferir a base

Abra no Colab. Se este notebook estiver numa pasta isolada, ele baixa a tabela publicada no repositório da disciplina. Confira 40 linhas, datas, URLs e o equilíbrio 10×4.

In [1]:
import pandas as pd
from pathlib import Path
arquivo_local = Path("dados/noticias_lula_flavio_2026.csv")
if not arquivo_local.exists():
    arquivo_local = Path("../dados/noticias_lula_flavio_2026.csv")
url_base = "https://raw.githubusercontent.com/mateuspestana/extracao_analise_2026/main/aulas/23-extracao-busca-semantica/dados/noticias_lula_flavio_2026.csv"
politica = pd.read_csv(arquivo_local if arquivo_local.exists() else url_base)
print(politica.shape)
print(pd.crosstab(politica["veiculo"], politica["foco_amostral"]))
assert len(politica) == 40

(40, 8)
foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                10    10
CNN Brasil                    10    10


## 1. Preparação textual — Aula 21

Preserve `texto_analise`; crie `texto_preparado` com regras justificadas. Conte documentos vazios, duplicatas, tokens antes/depois. Mostre três casos lado a lado; um deve conter nome próprio ou negação.

In [2]:
print("Vazios:", politica["texto_analise"].str.strip().eq("").sum())
print("Duplicatas exatas:", politica["texto_analise"].duplicated().sum())
print("Mediana de caracteres:", politica["texto_analise"].str.len().median())
print(politica.loc[0, "texto_analise"][:700])

Vazios: 0
Duplicatas exatas: 0
Mediana de caracteres: 203.5
Lula defende respeito às religiões no Dia de Nossa Senhora Aparecida. Ao citar o Minha Casa, Minha Vida, o presidente destacou que o tema da campanha da fraternidade da igreja Católica este ano é Fraternidade e Moradia.


In [3]:
import re
import unicodedata

def sem_acentos(texto):
    partes = unicodedata.normalize("NFD", texto)
    return "".join(letra for letra in partes if unicodedata.category(letra) != "Mn")

def preparar(texto, remover_acentos=False, remover_urls=True):
    texto = str(texto).lower()
    if remover_urls:
        texto = re.sub(r"https?://\S+", " ", texto)
    if remover_acentos:
        texto = sem_acentos(texto)
    tokens = re.findall(r"\b[^\W\d_]+\b", texto, flags=re.UNICODE)
    return " ".join(tokens)

politica["texto_preparado"] = politica["texto_analise"].map(preparar)
politica[["id", "foco_amostral", "texto_preparado"]].head(3)

,id,foco_amostral,texto_preparado
0,P01,Lula,lula defende respeito às religiões no dia de n...
1,P02,Lula,ato pela reeleição de lula leva milhares à ave...
2,P03,Lula,meta alega problemas técnicos para menor alcan...


In [4]:
for i in [4, 10, 28]:
    print("ID:", politica.loc[i, "id"])
    print("ANTES:", politica.loc[i, "texto_analise"][:180])
    print("DEPOIS:", politica.loc[i, "texto_preparado"][:180], "\n")

ID: P05
ANTES: Técnicos do TSE não encontram conteúdo falso no canal Exército do Lula. O setor de combate à desinformação do TSE avaliou os textos de mensagens e os vídeos, a pedido de Nunes Marq
DEPOIS: técnicos do tse não encontram conteúdo falso no canal exército do lula o setor de combate à desinformação do tse avaliou os textos de mensagens e os vídeos a pedido de nunes marque 

ID: P11
ANTES: Flávio passa domingo reunido com prefeitos e vereadores de São Paulo. Candidato do PL fez críticas ao presidente Lula e promessas econômicas. Pelas redes sociais, Flávio Bolsonaro 
DEPOIS: flávio passa domingo reunido com prefeitos e vereadores de são paulo candidato do pl fez críticas ao presidente lula e promessas econômicas pelas redes sociais flávio bolsonaro afi 

ID: P29
ANTES: Lula pavimenta fim da escala 6x1 em busca de "bala de prata" no 2º turno. Acordo com Alcolumbre, revelado pela CNN, mira vantagem nas urnas e freio à interferência dos EUA
DEPOIS: lula pavimenta fim da escala em

In [5]:
stopwords_pequenas = {"a", "o", "as", "os", "de", "da", "do", "das", "dos", "em", "um", "uma", "que", "e"}
frase = politica.loc[4, "texto_analise"]
print("Original:", frase)
print("Preparada:", preparar(frase))
print("Sem stopwords:", " ".join(t for t in preparar(frase).split() if t not in stopwords_pequenas))
print("Sem acentos:", preparar(frase, remover_acentos=True))

Original: Técnicos do TSE não encontram conteúdo falso no canal Exército do Lula. O setor de combate à desinformação do TSE avaliou os textos de mensagens e os vídeos, a pedido de Nunes Marques, após as decisões conflitantes sobre a liberação do canal.
Preparada: técnicos do tse não encontram conteúdo falso no canal exército do lula o setor de combate à desinformação do tse avaliou os textos de mensagens e os vídeos a pedido de nunes marques após as decisões conflitantes sobre a liberação do canal
Sem stopwords: técnicos tse não encontram conteúdo falso no canal exército lula setor combate à desinformação tse avaliou textos mensagens vídeos pedido nunes marques após decisões conflitantes sobre liberação canal
Sem acentos: tecnicos do tse nao encontram conteudo falso no canal exercito do lula o setor de combate a desinformacao do tse avaliou os textos de mensagens e os videos a pedido de nunes marques apos as decisoes conflitantes sobre a liberacao do canal


In [6]:
from collections import Counter
politica["tokens_antes"] = politica["texto_analise"].str.split().str.len()
politica["tokens_depois"] = politica["texto_preparado"].str.split().str.len()
print(politica[["tokens_antes", "tokens_depois"]].describe().round(1))
for foco in ["Lula", "Flávio Bolsonaro"]:
    texto = " ".join(politica.loc[politica.foco_amostral.eq(foco), "texto_preparado"])
    frequencias = Counter(t for t in texto.split() if t not in stopwords_pequenas)
    print(foco, frequencias.most_common(12))

       tokens_antes  tokens_depois
count          40.0           40.0
mean           32.0           31.6
std             7.1            7.2
min            17.0           17.0
25%            28.0           27.0
50%            32.5           32.0
75%            37.0           37.0
max            46.0           46.0
Lula [('lula', 23), ('no', 11), ('para', 11), ('é', 7), ('ato', 7), ('canal', 7), ('à', 6), ('flávio', 6), ('tse', 6), ('sobre', 6), ('com', 6), ('campanha', 5)]
Flávio Bolsonaro [('flávio', 24), ('lula', 12), ('bolsonaro', 12), ('candidato', 9), ('na', 8), ('para', 8), ('turno', 8), ('com', 7), ('ao', 7), ('pl', 6), ('à', 6), ('no', 6)]


A variável texto_analise foi conservado e texto_preparado foi criado com a função preparar. Há 0 textos vazios e 0 duplicatas exatas. A mediana é de 203,5 caracteres. A mediana de tokens vai de 32,5 para 32,0 e a média de 32,0 para 31,6. Os três casos lado a lado são P05 (nome próprio e negação), P11 (nome próprio composto e sigla) e P29 (números). Com a lista pequena de stopwords, com o foco em Lula, as palavras mais frequentes são lula (23), canal (7), ato (7) e tse (6), e no foco Flávio Bolsonaro são flávio (24), candidato (9), turno (8) e pl (6). Além disso, palavras como no, para, é também aparecem entre as mais frequentes. Para as regras e justificativas criadas foram: primeiramente, as minúsculas, para juntar Lula e lula. O custo disso foi perder a marca de nome próprio e de sigla. Assim, em P05, Nunes Marques virou nunes marques e em P11, PL virou pl. A segunda foi a remoção de URLs, regra mantida da aula. O texto de análise não tem URLs, então ela não teve efeito aqui. Terceiramente, os acentos foram mantidos, para preservar São Paulo e Flávio. A versão sem acentos aparece só na demonstração do P05, e o TF-IDF remove os acentos depois. Ademais pelo token ser uma sequência de letras, números e símbolos ficam fora, o que pode apagar algumas informações, como em P29, que escala 6x1 vira escala e 2º turno vira turno. Por fim, as stopwords não sairam da versão preparada, para preservar negações. Assim, a limpeza serve para a modelagem dos tópicos, mas perde números e a diferenciação entre primeiro e segundo turno, algo importante nesse recorte.

## 2. Tópicos — Aula 22

Aplique TF-IDF e NMF às 40 linhas. Use poucos tópicos e `random_state=42`. Mostre palavras e manchetes representativas, dê nomes provisórios e faça uma tabela por veículo e foco. Conte também as menções a Lula e Flávio nos títulos e resumos, sempre com o denominador de cada grupo. Explique ao menos um tópico confuso. As editorias do corpus geral da aula não existem nesta base.

In [7]:
import re
base = politica.drop_duplicates(subset="texto_analise").copy()
base = base.reset_index(drop=True)
base["texto_modelo"] = base["texto_analise"].str.lower().map(
    lambda s: " ".join(re.findall(r"\b[^\W\d_]+\b", s, flags=re.UNICODE))
)
print("Textos usados:", len(base), "de", len(politica))
print(base[["id", "veiculo", "foco_amostral"]].head())

Textos usados: 40 de 40
    id         veiculo foco_amostral
0  P01  Agência Brasil          Lula
1  P02  Agência Brasil          Lula
2  P03  Agência Brasil          Lula
3  P04  Agência Brasil          Lula
4  P05  Agência Brasil          Lula


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
stopwords = "a o e de da do das dos em no na nos nas um uma para por com que se os as ao foi ser são mais também lula flávio bolsonaro".split()
import unicodedata
stopwords = [unicodedata.normalize("NFD", w).encode("ascii", "ignore").decode("ascii") for w in stopwords]
vetor = TfidfVectorizer(strip_accents="unicode", stop_words=stopwords,
                        min_df=2, max_df=0.85, max_features=1000, ngram_range=(1, 2))
X = vetor.fit_transform(base["texto_modelo"])
print("Documentos x termos:", X.shape)
print("Termos de exemplo:", vetor.get_feature_names_out()[:15])

Documentos x termos: (40, 136)
Termos de exemplo: ['afirmou' 'alcolumbre' 'apoio' 'apoio campanha' 'apos' 'armadas' 'ato'
 'ato pela' 'ato rio' 'avenida' 'avenida paulista' 'campanha' 'canal'
 'canal exercito' 'candidato']


In [9]:
from sklearn.decomposition import NMF
import numpy as np
nmf = NMF(n_components=4, init="nndsvda", random_state=42, max_iter=300)
pesos = nmf.fit_transform(X)
termos = vetor.get_feature_names_out()
for i, componente in enumerate(nmf.components_):
    print(f"Tópico {i}:", ", ".join(termos[j] for j in componente.argsort()[-10:][::-1]))
base["topico_nmf"] = pesos.argmax(axis=1)
for i in range(4):
    posicoes = pesos[:, i].argsort()[-3:][::-1]
    print("\nTópico", i)
    for pos in posicoes:
        print(base.loc[pos, "id"], base.loc[pos, "veiculo"], base.loc[pos, "foco_amostral"], "|", base.loc[pos, "texto_analise"].splitlines()[0][:110])

Tópico 0: turno, segundo, segundo turno, senador, pai, candidato, presidencia, candidato presidencia, mira, dias
Tópico 1: canal, tse, exercito, canal exercito, eleitoral, sobre, nunes marques, marques, nunes, registrado
Tópico 2: ato, rio, apoio, defesa, ato rio, reeleicao, educacao, sociais, pl, quer
Tópico 3: campanha, casa, presidente, vida, site, oferece, site oferece, pretende, defende, tema

Tópico 0
P39 CNN Brasil Flávio Bolsonaro | Flávio: Bolsonaro está satisfeito com 1º turno e deu "dicas" para disputa. Senador e candidato à Presidência d
P38 CNN Brasil Flávio Bolsonaro | Flávio buscará orientação de Bolsonaro para reta final, dizem aliados. Senador e candidato à Presidência retom
P37 CNN Brasil Flávio Bolsonaro | Centrão mira pastas com maior orçamento se Flávio vencer. Partidos anunciaram apoio ao senador no segundo turn

Tópico 1
P05 Agência Brasil Lula | Técnicos do TSE não encontram conteúdo falso no canal Exército do Lula. O setor de combate à desinformação do 
P27 CNN

In [10]:
tabela_topicos = pd.crosstab([base["veiculo"], base["foco_amostral"]], base["topico_nmf"])
print(tabela_topicos)
base["peso_topico"] = pesos.max(axis=1)
print("Menor confiança relativa:")
print(base.nsmallest(5, "peso_topico")[["id", "veiculo", "foco_amostral", "topico_nmf", "peso_topico"]])

topico_nmf                       0  1  2  3
veiculo        foco_amostral               
Agência Brasil Flávio Bolsonaro  1  0  6  3
               Lula              0  3  4  3
CNN Brasil     Flávio Bolsonaro  6  2  1  1
               Lula              2  2  3  3
Menor confiança relativa:
     id         veiculo     foco_amostral  topico_nmf  peso_topico
39  P40      CNN Brasil  Flávio Bolsonaro           1     0.041261
23  P24      CNN Brasil              Lula           0     0.049869
22  P23      CNN Brasil              Lula           3     0.099920
16  P17  Agência Brasil  Flávio Bolsonaro           2     0.119386
35  P36      CNN Brasil  Flávio Bolsonaro           3     0.128009


In [11]:
base["mencoes_lula"] = base["texto_analise"].str.count(r"(?i)\bLula\b")
base["mencoes_flavio"] = base["texto_analise"].str.count(r"(?i)\bFlávio\b")
print(base.groupby(["veiculo", "foco_amostral"])[["mencoes_lula", "mencoes_flavio"]].sum())
print(pd.crosstab(base["veiculo"], base["foco_amostral"]))

                                 mencoes_lula  mencoes_flavio
veiculo        foco_amostral                                 
Agência Brasil Flávio Bolsonaro             6              13
               Lula                        13               4
CNN Brasil     Flávio Bolsonaro             6              11
               Lula                        10               2
foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                10    10
CNN Brasil                    10    10


In [12]:
tabela_topicos.to_csv("tabela_topicos.csv")
base.groupby(["veiculo", "foco_amostral"])[["mencoes_lula", "mencoes_flavio"]].sum().to_csv("tabela_mencoes.csv")

Primeiro, contei quantas vezes Lula e Flávio aparecem nos títulos e resumos, lembrando que cada grupo tem 10 matérias. Contei as vezes que o nome aparece, e não quantas matérias citam o nome. Na Agência Brasil, nas matérias com foco em Flávio, Lula aparece 6 vezes e Flávio 13. Nas com foco em Lula, Lula aparece 13 vezes e Flávio 4. Na CNN Brasil, no foco em Flávio, Lula aparece 6 vezes e Flávio 11, e no foco em Lula, Lula aparece 10 vezes e Flávio 2. Como as matérias já foram escolhidas por foco, era esperado que cada grupo citasse mais o candidato do seu foco. Então isso não mostra qual veículo dá mais espaço para cada um. Depois rodei o TF-IDF e o NMF com 4 tópicos, usando os 40 textos e tirando lula, flávio e bolsonaro, para os nomes não dominarem os tópicos. Chamei o tópico 0 de segundo turno e reta final de Flávio, por causa de palavras como turno, senador e pai e das manchetes P39, P38 e P37. O tópico 1 ficou como canal Exército do Lula e TSE. O tópico 2 ficou como atos de campanha e apoios. O tópico 3 eu não consegui nomear. Dá para ver que o T0 pega 6 das 10 matérias da CNN Brasil sobre Flávio e o T2 pega 10 das 20 da Agência Brasil. Já o tópico 3 é o confuso porque junta coisas que não têm a ver. Casa e vida vêm de duas matérias diferentes: a P01 fala do Minha Casa, Minha Vida e a P26 fala de custo de vida e de gravar campanha na casa de apoiadores. Já site e oferece vêm das matérias das criptomoedas. No fim, ele mistura 10 matérias dos quatro grupos sem um assunto em comum, e a P40 (0,041) e a P24 (0,050) ficaram em tópicos com peso bem baixo. Como são só 40 textos curtos e 4 tópicos, isso só mostra esse recorte e não prova que os veículos criam e publicam de um jeito diferente. As manchetes do T0 também são todas de 13/10, a 12 dias do segundo turno, então o tópico pode estar mostrando o dia e não um tema que se repete.

## 3. Extração estruturada — Aula 23

Crie sua própria chave Groq. Reaproveite o esquema e a função `extrair()` da aula principal. Selecione **4 notícias por combinação de veículo e foco** com `groupby(...).sample(n=4, random_state=42)`; são 16 chamadas ao todo. Faça primeiro **uma** chamada de teste. Depois rode o lote **uma vez** e guarde a tabela. Não repita o lote ao ajustar gráficos ou texto do relatório.

Confira se `evidencia` é um trecho literal de `texto_analise`. Audite manualmente pelo menos quatro fichas, uma por combinação. Marque ausências, exageros e atribuições ambíguas; não conserte silenciosamente a saída do modelo.

In [13]:
from getpass import getpass
chave_groq = getpass("Sua chave Groq (não aparece): ")
amostra_ia = politica.groupby(["veiculo", "foco_amostral"], group_keys=False).sample(n=4, random_state=42)
print(pd.crosstab(amostra_ia.veiculo, amostra_ia.foco_amostral))


Sua chave Groq (não aparece): ··········
foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                 4     4
CNN Brasil                     4     4


In [14]:
import requests, json
modelo_groq = "openai/gpt-oss-20b"
esquema = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "pessoa_central": {"type": ["string", "null"]},
        "fato": {"type": "string"},
        "tema": {"type": "string"},
        "evidencia": {"type": "string"},
        "limite": {"type": "string"}
    },
    "required": ["pessoa_central", "fato", "tema", "evidencia", "limite"]
}

def extrair(texto):
    resposta = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {chave_groq}"},
        json={"model": modelo_groq, "temperature": 0, "reasoning_effort": "low", "max_completion_tokens": 600,
              "messages": [{"role": "system", "content": "Extraia apenas fatos presentes no texto recebido. A evidencia deve ser um trecho literal e contínuo do texto. Se não houver pessoa central, use null. Não infira intenção política nem viés do veículo."},
                           {"role": "user", "content": texto}],
              "response_format": {"type": "json_schema", "json_schema": {"name": "ficha_noticia", "strict": True, "schema": esquema}}},
        timeout=40)
    if resposta.status_code == 429:
        raise RuntimeError("Limite Groq atingido (429). Aguarde antes de tentar outra chamada.")
    resposta.raise_for_status()
    return json.loads(resposta.json()["choices"][0]["message"]["content"])


In [15]:
teste = extrair(amostra_ia.iloc[0]["texto_analise"])
print(teste)
print("Evidência literal:", teste["evidencia"] in amostra_ia.iloc[0]["texto_analise"])


{'pessoa_central': 'Flávio', 'fato': 'TSE manda suspender site que oferece criptomoedas por apoio a Flávio', 'tema': 'eleições', 'evidencia': 'TSE manda suspender site que oferece criptomoedas por apoio a Flávio', 'limite': 'null'}
Evidência literal: True


In [16]:
fichas = []
for _, linha in amostra_ia.iterrows():
       resultado = extrair(linha["texto_analise"])
       fichas.append({"id": linha["id"], "veiculo": linha["veiculo"],
                    "foco_amostral": linha["foco_amostral"], **resultado,
                    "evidencia_literal": resultado["evidencia"] in linha["texto_analise"]})
fichas = pd.DataFrame(fichas)
fichas.to_csv("fichas_extraidas.csv", index=False)

In [17]:
print("Evidências literais:", fichas["evidencia_literal"].sum(), "de", len(fichas))
print(pd.crosstab(fichas.veiculo, fichas.evidencia_literal))

Evidências literais: 16 de 16
evidencia_literal  True
veiculo                
Agência Brasil        8
CNN Brasil            8


In [18]:
auditoria = fichas.groupby(["veiculo", "foco_amostral"], group_keys=False).sample(n=1, random_state=42)
for _, linha in auditoria.iterrows():
    texto = politica.loc[politica["id"].eq(linha["id"]), "texto_analise"].iloc[0]
    print("ID:", linha["id"], "|", linha["veiculo"], "|", linha["foco_amostral"])
    print("TEXTO:", texto)
    print("FICHA:", linha["pessoa_central"], "|", linha["fato"], "|", linha["tema"])
    print("EVIDÊNCIA:", linha["evidencia"], "| literal:", linha["evidencia_literal"], "\n")

ID: P12 | Agência Brasil | Flávio Bolsonaro
TEXTO: Flávio Bolsonaro participa de missa na Catedral de Brasília. Candidato disse que aguarda definição de horário para visitar o pai.
FICHA: Flávio Bolsonaro | Flávio Bolsonaro participa de missa na Catedral de Brasília. | religion
EVIDÊNCIA: Flávio Bolsonaro participa de missa na Catedral de Brasília. | literal: True 

ID: P02 | Agência Brasil | Lula
TEXTO: Ato pela reeleição de Lula leva milhares à Avenida Paulista. Candidato à reeleição disse que a semana será decisiva, por causa da votação do fim da escala 6x1, agendada para a próxima quarta-feira (14).
FICHA: Lula | Ato pela reeleição de Lula leva milhares à Avenida Paulista | Reeleição presidencial
EVIDÊNCIA: Ato pela reeleição de Lula leva milhares à Avenida Paulista | literal: True 

ID: P37 | CNN Brasil | Flávio Bolsonaro
TEXTO: Centrão mira pastas com maior orçamento se Flávio vencer. Partidos anunciaram apoio ao senador no segundo turno
FICHA: Flávio | Centrão mira pastas com ma

As 16 fichas foram geradas com o modelo openai/gpt-oss-20b, e em todas a evidência é um trecho do texto_analise (8 de cada veículo). Na chamada de teste, com a P19, o campo limite veio escrito "null" em vez de uma limitação de verdade. Para a auditoria, peguei uma ficha de cada combinação de veículo e foco (P12, P02, P37 e P28) e comparei com o texto_analise. Na P12 (Agência Brasil, foco no Flávio), a ficha usa só o título e deixa de fora o resumo, em que ele diz que aguarda o horário para visitar o pai. O tema veio como "religion", em inglês, enquanto o resto da ficha está em português. Na P02 (Agência Brasil, com foco no Lula), também só entrou o título, e ficou de fora o resumo, que fala da votação do fim da escala 6x1 marcada para 14/10. Na P37 (CNN Brasil, focada no Flávio), de novo só o título, e ficou de fora o apoio dos partidos ao senador no segundo turno. Aqui a pessoa central é o Flávio, mas quem age na frase é o Centrão, e o Flávio só aparece na condição "se vencer". O tema "Política" também ficou genérico demais. Na P28 (CNN Brasil, com foco no Lula), a ficha de novo só usa o título e deixa de fora a suspensão do grupo até o registro como canal de propaganda no TSE. A pessoa central é o Janones, que é quem está no título, mas o foco é o Lula, então a atribuição é só em parte clara. Assim, o padrão nas quatro é que o fato e a evidência só repetem o título e ignoram o resumo. Por isso a checagem de evidência literal é fraca, já que a ficha passa mesmo quando a evidência é só o título repetido. Dei uma olhada nas matérias, mas sem aprofundar por causa do tempo. Ainda falta abrir os links para ver se o resumo representa bem a matéria.

P12: https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/flavio-bolsonaro-participa-de-missa-na-catedral-de-brasilia-0

P02: https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/ato-pela-reeleicao-de-lula-leva-milhares-a-avenida-paulista

P37: https://www.cnnbrasil.com.br/blogs/isabel-mega/eleicoes/centrao-mira-pastas-com-maior-orcamento-se-flavio-vencer/

P28: https://www.cnnbrasil.com.br/eleicoes/janones-diz-que-cumprira-exigencias-do-tse-para-reativar-exercito-de-lula/

## 4. Busca — Aula 23

Faça uma pergunta substantiva. Compare top 3 de busca por palavra e de busca semântica nas 40 matérias. Registre IDs, títulos, links e uma avaliação manual da relevância. A consulta precisa poder ser repetida.

In [19]:
pergunta = "Quais reportagens mostram pesquisas de opinião da população sobre esses candidatos?"
vetor_consulta = vetor.transform([pergunta.lower()])
ranking_palavras = (X @ vetor_consulta.T).toarray().ravel().argsort()[-3:][::-1]
for pos in ranking_palavras:
    linha = politica.iloc[pos]
    print(linha["id"], linha["titulo"], linha["url"])

P08 PF nega existência de depoimento de Vorcaro sobre Lula https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/pf-nega-existencia-de-depoimento-envolvendo-vorcaro-e-lula
P33 O que sabemos sobre a propaganda de Flávio que foi exibida sem áudio https://www.cnnbrasil.com.br/eleicoes/o-que-sabemos-sobre-a-propaganda-de-flavio-que-foi-exibida-sem-audio/
P17 Mendonça multa Flávio por impulsionar post negativo contra Lula https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/mendonca-multa-flavio-por-impulsionar-post-negativo-contra-lula


In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import torch
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)
modelo_de_busca = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=dispositivo)
embeddings_noticias = modelo_de_busca.encode(politica["texto_analise"].tolist(), normalize_embeddings=True)
embedding_pergunta = modelo_de_busca.encode([pergunta], normalize_embeddings=True)
scores = cosine_similarity(embedding_pergunta, embeddings_noticias)[0]
for pos in scores.argsort()[-3:][::-1]:
    linha = politica.iloc[pos]
    print(linha["id"], round(float(scores[pos]), 3), linha["titulo"], linha["url"])

Dispositivo: cpu


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

P24 0.54 Gerp: Lula é rejeitado por 53% dos eleitores; Flávio, por 42% https://www.cnnbrasil.com.br/eleicoes/gerp-lula-e-rejeitado-por-53-dos-eleitores-flavio-por-42/
P21 0.502 MDB em dez estados do Norte e Nordeste declaram apoio à reeleição de Lula https://www.cnnbrasil.com.br/eleicoes/mdb-em-dez-estados-do-norte-e-nordeste-declaram-apoio-a-reeleicao-de-lula/
P14 0.463 Flávio Bolsonaro recebe apoio de governadores eleitos e Cury https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/flavio-bolsonaro-recebe-apoios-de-governadores-eleitos-e-cury


Fiz a busca com a pergunta "Quais reportagens mostram pesquisas de opinião da população sobre esses candidatos?". Ela dá para repetir, porque a pergunta fica guardada numa variável e os dois métodos não têm nada de aleatório. No recorte, as matérias que realmente tratam de pesquisa são a P23 e a P24, as duas do Gerp, na CNN Brasil. Na busca por palavra, o top 3 foi P08, P33 e P17, e nenhuma delas fala de pesquisa de opinião. A P08 é sobre a PF e o depoimento do Vorcaro, a P33 é sobre a propaganda do Flávio exibida sem áudio e a P17 é sobre a multa do Mendonça ao Flávio. Nas três aparece a palavra "sobre", e acho que foi isso que fez elas subirem. Já a P23 e a P24 usam "pesquisa" no singular, e a minha pergunta usa "pesquisas" no plural, então o TF-IDF pode não ter ligado uma coisa com a outra. Dessa forma, a busca semântica, o top 3 foi P24 (0,54), P21 (0,502) e P14 (0,463). A P24 é a pesquisa do Gerp que mostra Lula rejeitado por 53% e Flávio por 42%, se mostrou é relevante. Já a P21 é sobre o apoio do MDB à reeleição do Lula e a P14 é sobre o apoio de governadores eleitos ao Flávio, e nenhuma das duas tem pesquisa. A P23, que é a outra pesquisa, ficou fora do top 3 nos dois métodos. Resumindo, a busca por palavra achou 0 das 2 matérias de pesquisa e a semântica achou 1. Isso mostra o limite de comparar palavras exatas, sem lematização. Mas a semântica também trouxe duas matérias sem relação, com scores não tão distantes do acerto, e as duas são sobre apoio a candidatos, o que mostra que ela se confunde com o tema "candidatos". Como foi só uma pergunta e eu mesmo avaliei a relevância, isso serve para ilustrar os métodos e não para dizer qual é melhor.

P08: https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/pf-nega-existencia-de-depoimento-envolvendo-vorcaro-e-lula

P33: https://www.cnnbrasil.com.br/eleicoes/o-que-sabemos-sobre-a-propaganda-de-flavio-que-foi-exibida-sem-audio/

P17: https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/mendonca-multa-flavio-por-impulsionar-post-negativo-contra-lula

P24: https://www.cnnbrasil.com.br/eleicoes/gerp-lula-e-rejeitado-por-53-dos-eleitores-flavio-por-42/

P21: https://www.cnnbrasil.com.br/eleicoes/mdb-em-dez-estados-do-norte-e-nordeste-declaram-apoio-a-reeleicao-de-lula/

P14: https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/flavio-bolsonaro-recebe-apoios-de-governadores-eleitos-e-cury

P23: https://www.cnnbrasil.com.br/eleicoes/gerp-49-dos-brasileiros-desaprovam-governo-lula-45-aprovam/

## 5. Relatório e entrega

No `README.md`, siga esta ordem: objetivo e pergunta; preparação; revisão de 21–22; roteiro de análise; prática aplicada; resultados do desafio; entrega; declaração de uso de IA; checklist. Inclua fonte, janela temporal, regra de seleção, tabela de denominadores, pelo menos dois achados com IDs/URLs, quatro fichas auditadas, tópico confuso, comparação de buscas e limitações. Separe **descrição dos dados** de **interpretação**. Não chame a diferença observada de “viés do jornal” sem um protocolo mais amplo.

Faça um commit de progresso e um de entrega. Prepare-se para explicar uma decisão de limpeza, um tópico, uma ficha e uma busca.

- [ ] 40 linhas: 10 por veículo × foco.
- [ ] Chave pessoal fora dos arquivos entregues.
- [ ] Extração em amostra de 16, com quatro auditorias.
- [ ] Conclusões citam IDs e links e respeitam o recorte.
- [ ] Notebook executa de cima a baixo no Colab.